# CN-GongWen Benchmark · 一键**完美复现** Colab（四套公文评测数据集）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/SH-PA/blob/main/notebooks/CN_GongWen_Reproduce_Colab.ipynb)

端到端生成并校验**中国党政机关公文**评测数据（对标《党政机关公文处理工作条例》与 GB/T 9704—2012，约一半医疗卫生方向）：

| 数据集 | 内容 | 打分 |
|---|---|---|
| **CN-GongWen-Q** | 纯问题集，15 类题型（含文种误用/行文关系/权限边界/**医疗合规**辨析） | `--dataset q` |
| **CN-GongWen-DataQA** | 合成语料数据问答，16 类任务 + 医疗三级分类 | `--dataset dataqa` |
| **CN-GongWen-Writing** | 按 token 分桶的写作测试 prompt，11 维结构化打分（含医学合规） | `--dataset writing` |
| **CN-GongWen-Audit** | 公文审核/纠错：找出 16 类违规 + 改写为合规公文 | `--dataset audit` / `rewrite` |

**为什么"完美复现"**：数据生成完全基于 `SHA-256`、**无随机数、核心无第三方依赖**，任何环境/时间从源码重生成都与提交工件**逐字节一致**——第 6 步用 `git diff` 当场证明。

> ⚠️ 所有机关、姓名、字号、内容均为**合成示例**，不对应任何真实单位或公文，不含个人隐私。


## 0️⃣ 配置

In [ ]:
REPO_URL = 'https://github.com/pariskang/SH-PA.git'
BRANCH   = 'main'                  # 已合并至 main；如需特定分支自行修改
PROFILE  = 'standard'              # mini(快) / standard(提交基准) / full(生产规模)
print('repo =', REPO_URL, '| branch =', BRANCH, '| profile =', PROFILE)

## 1️⃣ 环境信息（可复现性记录）

In [ ]:
import sys, platform, datetime
print('Python  :', sys.version.split()[0])
print('Platform:', platform.platform())
print('Time    :', datetime.datetime.now().isoformat(timespec='seconds'))
print('注：核心生成仅依赖标准库，结果与上述环境无关（确定性）。')

## 2️⃣ 克隆仓库（指定分支，可重复运行）

In [ ]:
import os
BASE = '/content' if os.path.isdir('/content') else os.getcwd()
os.chdir(BASE)
if not os.path.isdir('SH-PA'):
    !git clone --branch {BRANCH} {REPO_URL} SH-PA
os.chdir(os.path.join(BASE, 'SH-PA'))
!git checkout -q {BRANCH} 2>/dev/null; git pull -q origin {BRANCH} 2>/dev/null
print('CWD:', os.getcwd())
!git log --oneline -3

## 3️⃣ 安装依赖（含中文字体）

核心生成/校验仅需标准库；此处装 `pytest`、`PyYAML` 以便测试，并装 Noto CJK 字体以正确显示图表中文。

In [ ]:
!pip -q install pytest PyYAML
!apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1 || true
# 可选：LLM 改写冻结 / Parquet 导出
# !pip -q install 'litellm>=1.0.0' openai pandas pyarrow
print('依赖安装完成')

## 4️⃣ 配置中文字体（修复图表中文乱码）

In [ ]:
import glob, matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
cjk = None
for pat in ('NotoSansCJK*', 'NotoSerifCJK*', 'wqy*', '*CJK*'):
    for p in glob.glob(f'/usr/share/fonts/**/{pat}', recursive=True):
        try:
            fm.fontManager.addfont(p); cjk = fm.FontProperties(fname=p).get_name(); break
        except Exception:
            continue
    if cjk: break
if cjk:
    plt.rcParams['font.sans-serif'] = [cjk, 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('中文字体：', cjk or '未找到（图表中文可能为方块，可重跑第 3、4 步）')

## 5️⃣ （可选）配置 MiniMax —— LLM 改写并"冻结"

`--use-litellm` 仅在**事实护栏**下改写**表层文本**：Q 问题措辞 / DataQA 播报语言 / Writing 测试 prompt；
文种、字号、日期、密级、数值、排序、合规判定，以及写作 rubric / 参考公文 / 审核金标准**始终由 Python 确定性计算**。
单条调用失败自动回退确定性模板（冻结流程不中断）。运行后 `git add` 提交即"冻结"这批 LLM 产物。不配置可直接跳过。

In [ ]:
import os
# os.environ['MINIMAX_API_KEY']  = 'sk-...'
# os.environ['MINIMAX_API_BASE'] = 'https://api.minimaxi.com/v1'
# os.environ['MINIMAX_MODEL']    = 'MiniMax-M1'
USE_LLM = bool(os.getenv('MINIMAX_API_KEY'))
print('使用 LiteLLM 改写并冻结：', USE_LLM)

## 6️⃣ 一键生成四套数据集，并**当场证明逐字节一致**

In [ ]:
import subprocess
cmd = f'python gongwen_benchmark/scripts/generate_benchmarks.py --profile {PROFILE}'
if USE_LLM: cmd += ' --use-litellm'
print('$', cmd)
!{cmd}
print()
if PROFILE == 'standard' and not USE_LLM:
    diff = subprocess.run(['git','status','--porcelain','gongwen_benchmark'], capture_output=True, text=True).stdout.strip()
    print('✅ 完美复现：四套数据集与已提交 standard 版逐字节一致。' if not diff
          else '⚠ 与提交版存在差异：\n' + diff[:1000])
else:
    print(f'profile={PROFILE} 或启用了 LLM，跳过与提交基准(standard)的逐字节比对。')

## 7️⃣ 严格跨文件校验（四套数据集）

In [ ]:
import json, subprocess
rep = subprocess.run(['python','gongwen_benchmark/scripts/validate_artifacts.py'], capture_output=True, text=True)
if rep.returncode != 0 or not rep.stdout.strip():
    print('❌ 校验未通过：'); print(rep.stderr.strip() or '(无输出)'); raise SystemExit(1)
report = json.loads(rep.stdout)
for k in ['q','dataqa','records','writing','audit',
          'q_hard_share','q_medical_share','q_trap_diversity',
          'corpus_medical_share','medical_area_coverage','medical_topic_coverage',
          'writing_buckets','writing_reference_in_range_share',
          'audit_clean','audit_flawed','audit_violation_coverage']:
    print(f'  {k:30s}: {report[k]}')
print('✓ 严格跨文件校验通过')

## 8️⃣ 单元测试

In [ ]:
!pytest -q

## 9️⃣ CN-GongWen-Q / DataQA 维度分布

In [ ]:
import json, collections
ROOT = 'gongwen_benchmark'
def load(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
hidden = load(f'{ROOT}/dataset_1_question_only/questions_with_hidden_metadata.jsonl')
ques   = load(f'{ROOT}/dataset_2_data_qa/questions.jsonl')
qt = collections.Counter(x['question_type'] for x in hidden)
tt = collections.Counter(x['task_type'] for x in ques)
df = collections.Counter(x['difficulty'] for x in hidden)
fig, ax = plt.subplots(1, 3, figsize=(20, 6))
ax[0].barh(list(qt.keys()), list(qt.values())); ax[0].set_title('CN-GongWen-Q 问题类型(15)')
ax[1].barh(list(tt.keys()), list(tt.values())); ax[1].set_title('CN-GongWen-DataQA 任务类型(16)')
ax[2].bar(list(df.keys()), list(df.values())); ax[2].set_title('Q 难度分布')
plt.tight_layout(); plt.show()
print('hard 占比：', round(df['hard']/sum(df.values()), 3), '| 题型数：', len(qt))

## 🔟 医疗政策**三级分类**可视化

政策领域（约半医疗）→ 16 个医疗子领域 → 约 105 个具体政策主题。

In [ ]:
import csv
recs = list(csv.DictReader(open(f'{ROOT}/dataset_2_data_qa/records.csv', encoding='utf-8')))
dom  = collections.Counter(r['policy_domain'] for r in recs)
qdom = collections.Counter(x['policy_domain'] for x in hidden)
area = collections.Counter(r['medical_area'] for r in recs if r['medical_area'])
topic= collections.Counter(r['medical_topic'] for r in recs if r['medical_topic'])
fig, ax = plt.subplots(1, 3, figsize=(20, 7))
ax[0].bar(['语料·'+k for k in dom]+['Q·'+k for k in qdom], list(dom.values())+list(qdom.values()))
ax[0].set_title('政策领域分布（各约50%医疗）'); ax[0].tick_params(axis='x', rotation=30)
ax[1].barh(list(area.keys()), list(area.values())); ax[1].set_title('16 个医疗子领域·语料发文量')
top = topic.most_common(20)[::-1]
ax[2].barh([t for t,_ in top], [c for _,c in top]); ax[2].set_title('具体政策主题 Top20（共%d个）' % len(topic))
plt.tight_layout(); plt.show()
print('语料医疗占比：', round(dom['医疗卫生']/sum(dom.values()),3), '| Q医疗占比：', round(qdom['医疗卫生']/sum(qdom.values()),3),
      '| 子领域：', len(area), '| 具体主题：', len(topic))

## 1️⃣1️⃣ DataQA 证据行复核（全部可追溯）

In [ ]:
doc_ids = {r['doc_id'] for r in recs}
ans = load(f'{ROOT}/dataset_2_data_qa/answers.jsonl')
bad = [a['question_id'] for a in ans if not set(a['evidence_rows']) <= doc_ids]
print('记录数：', len(doc_ids), '| 答案数：', len(ans), '| 证据越界题：', len(bad))
assert not bad, bad[:5]
print('✓ DataQA 证据行全部可追溯')

## 1️⃣2️⃣ CN-GongWen-Writing：按 token 分桶的写作测试 prompt

短/中/长按**目标产出 token** 分桶，覆盖 15 文种；参考公文均落入目标 token 区间。

In [ ]:
from pathlib import Path
from gongwen_benchmark.scripts.tokens import estimate_tokens
from gongwen_benchmark.scripts.generate_writing_prompts import LENGTH_BUCKETS as LB
wrub = load(f'{ROOT}/dataset_3_writing/writing_prompts_with_rubric.jsonl')
order = ['short', 'medium', 'long']
wbucket = collections.Counter(h['length_bucket'] for h in wrub)
wdt = collections.Counter(h['spec']['doc_type'] for h in wrub)
wtok = {b: [estimate_tokens(h['reference_answer']) for h in wrub if h['length_bucket']==b] for b in order}
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].bar([f"{b}\n{LB[b]['target_tokens']}" for b in order], [wbucket[b] for b in order]); ax[0].set_title('Writing 长度分桶（目标产出 token）')
ax[1].barh(list(wdt.keys()), list(wdt.values())); ax[1].set_title('Writing·15 文种覆盖')
ax[2].boxplot([wtok[b] for b in order]); ax[2].set_xticks([1,2,3]); ax[2].set_xticklabels(order); ax[2].set_title('参考公文 token 分布（按桶）')
plt.tight_layout(); plt.show()
print('Writing 题量：', len(wrub), '| 桶：', dict(wbucket), '| 医疗占比：', round(sum(h['spec']['policy_domain']=='医疗卫生' for h in wrub)/len(wrub),3))

## 1️⃣3️⃣ CN-GongWen-Audit：公文审核/纠错（找错 + 改写）

含缺陷公文（约 1/4 合规对照），覆盖 16 类违规（含 5 类医疗专属：夸大疗效、患者隐私、科研当临床、AI 替代医师、医保违规）。

In [ ]:
ag = load(f'{ROOT}/dataset_4_audit/audit_tasks_with_gold.jsonl')
vc = collections.Counter(v for h in ag for v in h['violations'])
clean = sum(h['is_clean'] for h in ag)
fig, ax = plt.subplots(1, 2, figsize=(20, 6))
ax[0].bar(['合规对照','含缺陷'], [clean, len(ag)-clean]); ax[0].set_title('Audit 样本：合规 vs 含缺陷')
items = vc.most_common()[::-1]
ax[1].barh([k for k,_ in items], [c for _,c in items]); ax[1].set_title('16 类违规分布（含 5 类医疗专属）')
plt.tight_layout(); plt.show()
print('Audit 题量：', len(ag), '| 合规：', clean, '| 含缺陷：', len(ag)-clean, '| 违规类型覆盖：', len(vc))

## 1️⃣4️⃣ 五个打分器·金标准自评（应全部 = 1.0）

In [ ]:
from gongwen_benchmark.evaluation.scorer import (
    dataset1_score, dataset2_score, dataset3_writing_score, dataset4_audit_score, dataset4_rewrite_score)
R = Path(ROOT)
d1 = R/'dataset_1_question_only/questions_with_hidden_metadata.jsonl'
d2 = R/'dataset_2_data_qa/answers.jsonl'
d3 = R/'dataset_3_writing/writing_prompts_with_rubric.jsonl'
d4 = R/'dataset_4_audit/audit_tasks_with_gold.jsonl'
print('Q       (文种路由) :', round(dataset1_score(d1, d1)['doc_type_routing_accuracy'], 3))
print('DataQA  (答案值)   :', round(dataset2_score(d2, d2)['answer_value_accuracy'], 3))
print('Writing (总体合规) :', round(dataset3_writing_score(d3, d3)['overall_compliance'], 3))
print('Audit·找错 (F1)    :', round(dataset4_audit_score(d4, d4)['violation_f1'], 3))
print('Audit·改写 (总体)  :', round(dataset4_rewrite_score(d4, d4)['overall_rewrite_compliance'], 3))

## 1️⃣5️⃣ 样例速览（医疗合规 Q / 写作 prompt / 审核找错+改写）

In [ ]:
mc = next(h for h in hidden if h['question_type'] == 'medical_compliance')
print('【医疗合规 Q】', mc['question'][:90], '...')
wm = next(h for h in wrub if h['spec']['policy_domain'] == '医疗卫生')
print('\n【写作测试 prompt·医疗（节选）】\n', wm['prompt'][:200], '...')
af = next(h for h in ag if not h['is_clean'])
print('\n【审核·含缺陷公文】注入违规：', '、'.join(af['violations']))
for l in af['flawed_document'].splitlines():
    if any(k in l for k in ('确保治愈','患者张某','无需医生审核','分解住院','全面推广','[20','第99条','（一）、')):
        print('   缺陷行:', l.strip())
print('   → 合规改写（金标准）标题:', next((l for l in af['corrected_document'].splitlines() if '关于' in l), '(无)'))

## ✅ 完美复现完成

你已从源码端到端重建并校验**四套**公文评测数据集，并验证了与提交基准的**逐字节一致性**。

用自己的模型评分：把预测写成与金标准同 `question_id` 的 JSONL，运行对应打分器：

```bash
# 1) CN-GongWen-Q   预测: {"question_id","target_doc_type","expected_query_type","requires_clarification","should_refuse"}
python gongwen_benchmark/evaluation/scorer.py --dataset q \
  --gold gongwen_benchmark/dataset_1_question_only/questions_with_hidden_metadata.jsonl --pred your_q.jsonl
# 2) CN-GongWen-DataQA  预测: {"question_id","answer_value","evidence_rows"}
python gongwen_benchmark/evaluation/scorer.py --dataset dataqa \
  --gold gongwen_benchmark/dataset_2_data_qa/answers.jsonl --pred your_dataqa.jsonl
# 3) CN-GongWen-Writing 预测: {"question_id","answer":"<公文全文>"}
python gongwen_benchmark/evaluation/scorer.py --dataset writing \
  --gold gongwen_benchmark/dataset_3_writing/writing_prompts_with_rubric.jsonl --pred your_writing.jsonl
# 4) CN-GongWen-Audit·找错  预测: {"question_id","violations":["code",...]}
python gongwen_benchmark/evaluation/scorer.py --dataset audit \
  --gold gongwen_benchmark/dataset_4_audit/audit_tasks_with_gold.jsonl --pred your_audit.jsonl
# 5) CN-GongWen-Audit·改写  预测: {"question_id","rewrite":"<改写后的公文全文>"}
python gongwen_benchmark/evaluation/scorer.py --dataset rewrite \
  --gold gongwen_benchmark/dataset_4_audit/audit_tasks_with_gold.jsonl --pred your_rewrite.jsonl
```

> **用 LLM 撰写并冻结**：配置 `MINIMAX_API_KEY` 后在第 5、6 步启用 `--use-litellm`，
> 即由 LLM 在事实护栏下改写问题/播报/写作 prompt（rubric 与审核金标准仍为确定性），`git add` 提交即冻结。
